In [ ]:
import pandas as pd

datasets = {
    'Synthetic': '../data/herg_data/herg_ecfp_linear.csv',
    'CNOHF': '../data/cnohf_data/cnohf_ecfp.csv',
    'Photoswitch': '../data/photoswitch_data/photoswitch_ecfp.csv',
    'Polymers': '../data/polymers_data/polymers_ecfp.csv',
    'Redox': '../data/redox_data/redox_ecfp.csv',
    'COF': '../data/cof_data/cof_ecfp_descriptor.csv',
}

dataset_df = {
    'Dataset': [],
    'Num Samples': [],
    'Num ECFP': [],
    'Num Other Features': [],
}
for name, path in datasets.items():
    df = pd.read_csv(path)
    dataset_df['Dataset'].append(name)
    dataset_df['Num Samples'].append(df.shape[0])
    num_ecfp = sum(col.startswith('ecfp') for col in df.columns)
    dataset_df['Num ECFP'].append(num_ecfp)
    dataset_df['Num Other Features'].append(df.shape[1] - num_ecfp - 2)

dataset_summary = pd.DataFrame(dataset_df)
print(dataset_summary)

print(dataset_summary.to_latex())


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

datasets = {
    'Synthetic Linear': ['../data/herg_data/herg_ecfp_linear.csv', 'target'],
    'Synthetic Piecewise': ['../data/herg_data/herg_ecfp_piecewise.csv', 'target'],
    'Synthetic Polynomial': ['../data/herg_data/herg_ecfp_nonlinear.csv', 'target'],
    'CNOHF': ['../data/cnohf_data/cnohf_ecfp.csv', 'detonation_velocity'],
    'Photoswitch': ['../data/photoswitch_data/photoswitch_ecfp.csv', 'e_isomer_pi_pi'],
    'Polymers': ['../data/polymers_data/polymers_ecfp.csv', 'Tg'],
    'Redox': ['../data/redox_data/redox_ecfp.csv', 'dGox'],
    'COF': ['../data/cof_data/cof_ecfp_descriptor.csv', 'capacity_max'],
}

target_col_mapping = {
    'target': 'Target',
    'capacity_max': 'Capacitance (F/g)',
    'dGox': 'Gibbs free energy variation of oxidation (kcal mol$^{-1}$)',
    'detonation_velocity': 'Detonation velocity (km/s)',
    'e_isomer_pi_pi': 'E-isomer $\\pi$-$\\pi^*$ transition wavelength (nm)',
    'Tg': 'Glass transition temperature (K)',
}

def visualize_targets(datasets, target_col_mapping):
    plt.style.use('default')
    layout = [
        ["A", "A", "B", "B", "C", "C"],  # Row 1 (3 plots)
        ["D", "D", "E", "E", "F", "F"],  # Row 2 (3 plots)
        [".", "G", "G", "H", "H", "."]   # Row 3 (2 plots, centered)
    ]
    axes_keys = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    fig, axes = plt.subplot_mosaic(layout, figsize=(14, 10), constrained_layout=True)

    for i, (name, t) in enumerate(datasets.items()):
        p, target_col = t
        df = pd.read_csv(p)
        print(name, t)
        sns.histplot(data=df, x=target_col, bins=50, kde=True, ax=axes[axes_keys[i]])
        axes[axes_keys[i]].set_title(name)
        axes[axes_keys[i]].set_xlabel(target_col_mapping[target_col])
        axes[axes_keys[i]].set_ylabel('Frequency')

    plt.savefig("molecules/targets.pdf", bbox_inches='tight', facecolor='white', transparent=False)
    plt.show()

visualize_targets(datasets, target_col_mapping)

In [ ]:
import io
from io import BytesIO

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image
from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor, rdAbbreviations
from rdkit.Chem.Draw import MolDrawOptions

from src.core.utils import select_diverse_complex_subset, select_diverse_subset_butina

def get_diverse_mols(smiles_list, num_examples=5):
    smiles_ids = select_diverse_complex_subset(
        [Chem.MolFromSmiles(smiles) for smiles in smiles_list],
        num_to_select=num_examples, similarity_cutoff=0.3
    )
    mols = [Chem.MolFromSmiles(smiles_list[i]) for i in smiles_ids]
    return mols


def visualize_molecules(mols, per_row=5, subim_size=(200,200)):
    options = MolDrawOptions()
    options.padding = 0.01
    img = Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=subim_size, drawOptions=options)

    png_data = img.data
    img_arr = mpimg.imread(io.BytesIO(png_data), format='png')

    mask = np.any(img_arr[:, :, :3] != 1, axis=2)

    # Find the rows and columns that contain content
    rows_with_content = np.any(mask, axis=1)
    cols_with_content = np.any(mask, axis=0)

    # Find the min/max indices of the content
    if np.any(rows_with_content):
        ymin, ymax = np.where(rows_with_content)[0][[0, -1]]
        xmin, xmax = np.where(cols_with_content)[0][[0, -1]]

        # Add a tiny 1-pixel border for aesthetics
        ymin = max(0, ymin - 1)
        ymax = min(img_arr.shape[0], ymax + 2)
        xmin = max(0, xmin - 1)
        xmax = min(img_arr.shape[1], xmax + 2)

        # 4. Slice the array to the content box
        cropped_arr = img_arr[ymin:ymax, xmin:xmax]
        return cropped_arr
    else:
        # Return the original image if it's all white (or empty)
        return img_arr

from rdkit import Chem
from rdkit.Chem import Draw

def visualize_molecules_svg(mols, filename, per_row=5, subim_size=(200,200)):
    # 1. Setup options
    options = Draw.MolDrawOptions()
    options.padding = 0.05

    prepared_mols = []
    for mol in mols:
        rdDepictor.Compute2DCoords(mol)
        abbrevs = rdAbbreviations.GetDefaultAbbreviations()
        nm = rdAbbreviations.CondenseMolAbbreviations(mol, abbrevs)
        prepared_mols.append(nm)

    # 2. Generate SVG (Note: the parameter is 'useSVG', not 'returnSVG')
    svg_data = Draw.MolsToGridImage(
        prepared_mols,
        molsPerRow=per_row,
        subImgSize=subim_size,
        drawOptions=options,
        useSVG=True
    )
    svg_data = svg_data.data
    # 3. Save to file
    if filename:
        with open(filename, "w") as f:
            f.write(svg_data)

    return svg_data

In [ ]:
import os
os.makedirs('molecules', exist_ok=True)

for name, path in datasets.items():
    df = pd.read_csv(path)
    smiles_list = df['smiles'].tolist()
    mol_list = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
    mols_ids = select_diverse_subset_butina(mol_list, num_to_select=6, similarity_cutoff=0.3)
    mols = [Chem.MolFromSmiles(smiles_list[i]) for i in mols_ids]
    visualize_molecules_svg(mols, filename=f'molecules/{name}_examples.svg', per_row=3, subim_size=(400,400))

In [ ]:
df = pd.read_csv('../data/cof_data/cof_ecfp_descriptor.csv')
smiles_list = df['smiles'].tolist()
mol_list = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
mols_ids = [0, 17, 24, 31, 65, 7]
mols = [Chem.MolFromSmiles(smiles_list[i]) for i in mols_ids]
visualize_molecules_svg(mols, filename=f'molecules/COF_examples.svg', per_row=3, subim_size=(400,400))
